# Análisis Exploratorio de Datos (EDA) — Churn de Clientes

Este notebook explora el dataset de clientes para entender qué factores se relacionan con el abandono (*churn*) antes de construir el modelo predictivo.

**Objetivo:** identificar las *fricciones* que afectan la experiencia del cliente y aumentan el riesgo de abandono.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv('../data/clientes.csv')
print(f'Total de clientes: {len(df):,}')
df.head()

## 1. Vista general y calidad de los datos

In [ ]:
df.info()
print('\nValores nulos por columna:')
print(df.isnull().sum())

In [ ]:
df.describe()

## 2. ¿Cuál es la tasa de churn general?

In [ ]:
tasa = df['churn'].mean()
print(f'Tasa de churn: {tasa:.1%}')
fig = px.pie(df, names='churn_label', title='Distribución de churn', hole=0.4)
fig.show()

## 3. Churn por tipo de contrato

Hipótesis: los contratos mensuales tienen mayor abandono porque no hay compromiso de permanencia.

In [ ]:
churn_contrato = df.groupby('tipo_contrato')['churn'].mean().reset_index()
fig = px.bar(churn_contrato, x='tipo_contrato', y='churn',
             title='Tasa de churn por tipo de contrato', color='churn',
             color_continuous_scale='Reds')
fig.update_layout(yaxis_tickformat='.0%')
fig.show()

## 4. Relación entre antigüedad y churn

In [ ]:
fig = px.box(df, x='churn_label', y='meses_antiguedad',
             title='Antigüedad vs churn', color='churn_label')
fig.show()
print('Antigüedad promedio:')
print(df.groupby('churn_label')['meses_antiguedad'].mean())

## 5. Impacto de los tickets de soporte (fricción)

In [ ]:
churn_tickets = df.groupby('tickets_soporte')['churn'].mean().reset_index()
fig = px.line(churn_tickets, x='tickets_soporte', y='churn', markers=True,
              title='Tasa de churn según número de tickets de soporte')
fig.update_layout(yaxis_tickformat='.0%')
fig.show()

## Conclusiones del EDA

### Hallazgos principales

| # | Hallazgo | Evidencia |
|---|---|---|
| 1 | **Contratos mensuales** → mayor riesgo | Tasa de churn más alta en el gráfico por contrato |
| 2 | **Baja antigüedad** → más abandono | Box plot: clientes que se van tienen mediana de antigüedad menor |
| 3 | **Tickets de soporte** → señal de fricción | La tasa de churn sube con cada ticket adicional |
| 4 | **Tipo de contrato y método de pago** → asociación estadística confirmada | Chi-cuadrado con p < 0.05 |
| 5 | **Cargo mensual y antigüedad** correlacionan con churn | Matriz de correlación |
| 6 | **Género y presencia de dependientes** → sin asociación significativa con churn | Chi² con p > 0.05 |

### Implicaciones para el modelo
- Las variables con mayor chi² son los mejores candidatos a features predictivos.
- La **correlación entre cargo_mensual y cargo_total** (~0.7+) sugiere multicolinealidad; el modelo necesita regularización o selección (RandomForest y Gradient Boosting son robustos a esto; Regresión Logística menos).
- La baja correlación lineal del churn (~0.2–0.3 máx.) indica que el problema requiere modelos no lineales o combinaciones de features.

Estos hallazgos guían el modelo predictivo en `src/train_model.py`.

In [ ]:
num_cols = ['edad', 'meses_antiguedad', 'cargo_mensual', 'cargo_total', 'tickets_soporte', 'churn']
corr = df[num_cols].corr()

fig = px.imshow(
    corr,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Matriz de correlación — variables numéricas',
)
fig.update_layout(width=600, height=500)
fig.show()

print('Correlación con churn (descendente):')
print(corr['churn'].drop('churn').sort_values(ascending=False).to_string())

## 7. Distribución de variables numéricas por estado de churn

Los **violin plots** combinan un box plot (mediana, cuartiles) con la distribución de densidad. Permiten ver no solo la mediana sino la forma completa de la distribución para clientes que se fueron vs. los que se quedaron.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

vars_num = ['cargo_mensual', 'meses_antiguedad', 'tickets_soporte', 'edad']
fig = make_subplots(rows=2, cols=2, subplot_titles=vars_num)

for i, var in enumerate(vars_num):
    row, col = divmod(i, 2)
    for label, color in [('Si', '#EF553B'), ('No', '#636EFA')]:
        fig.add_trace(
            go.Violin(
                y=df[df['churn_label'] == label][var],
                name=f'Churn={label}',
                box_visible=True,
                meanline_visible=True,
                fillcolor=color,
                opacity=0.6,
                line_color=color,
                showlegend=(i == 0),
            ),
            row=row + 1, col=col + 1,
        )

fig.update_layout(
    title='Distribución de variables numéricas por estado de churn',
    height=600,
    violinmode='group',
)
fig.show()

# Resumen estadístico por grupo
for var in vars_num:
    print(f'\n{var}:')
    print(df.groupby('churn_label')[var].agg(['mean', 'median']).round(2))

## 8. Factores categóricos: test chi-cuadrado

El **test chi-cuadrado** mide si la distribución de una variable categórica es independiente del churn. Un p-value < 0.05 indica que la asociación no es aleatoria — esa variable contiene señal predictiva.

In [ ]:
from scipy.stats import chi2_contingency

cat_vars = [
    'tipo_contrato', 'metodo_pago', 'servicio_internet', 'soporte_tecnico',
    'factura_electronica', 'genero', 'tiene_pareja', 'tiene_dependientes',
]

resultados_chi2 = []
for var in cat_vars:
    tabla = pd.crosstab(df[var], df['churn'])
    chi2, p_value, dof, _ = chi2_contingency(tabla)
    resultados_chi2.append({
        'variable': var,
        'chi2': round(chi2, 2),
        'p_value': round(p_value, 4),
        'significativa (p<0.05)': 'Sí ✓' if p_value < 0.05 else 'No ✗',
    })

chi2_df = pd.DataFrame(resultados_chi2).sort_values('p_value')
print('Test chi-cuadrado: asociación con churn\n')
print(chi2_df.to_string(index=False))

# Visualización
fig = px.bar(
    chi2_df, x='chi2', y='variable', orientation='h',
    color='p_value', color_continuous_scale='RdYlGn_r',
    title='Estadístico chi² por variable categórica (mayor = más asociación con churn)',
    labels={'chi2': 'Chi²', 'variable': ''},
)
fig.update_layout(showlegend=False)
fig.show()

## Conclusiones

- Los **contratos mensuales** concentran el mayor riesgo de abandono.
- Los clientes con **baja antigüedad** abandonan más → importancia del onboarding.
- Un mayor número de **tickets de soporte** se asocia con mayor churn → la fricción operativa impacta la retención.

Estos hallazgos guían el modelo predictivo en `src/train_model.py`.